# Lesson 11 Exercise: Pack sparse connectivity for sequential lookup

This workbook corresponds to **Lesson 11: Why not scan every synapse after every spike?**

You will transform an edge list into source_index + contiguous records. The real goal is not writing a nested loop; it is understanding **how one source directly locates its contiguous synapse range**.

## Data contract

Each input edge is (source, target, weight).

Output:

- source_index[source] = (start, count)
- records stores contiguous (target, weight) entries

Requirements:

- source IDs range from 0 to num_sources-1;
- preserve input edge order within each source;
- a zero-fanout source still has a valid (start, 0);
- lookup_source() returns exactly that source's contiguous slice.

## Before coding: pack a small graph by hand

Use num_sources = 3 and these edges in this order:

- (0, 2, +5)
- (2, 0, -1)
- (2, 1, +4)

Write:

1. the order of records;
2. source 0's (start, count);
3. source 1's (start, count), even though it has no outgoing edge;
4. source 2's start and count;
5. the slices for sources 0, 1, and 2.

If the manual layout is not clear yet, do not start build_source_index().

## Implementation planning before code

This is the first exercise that combines a data layout with a construction algorithm, so break the task down without turning it into copyable pseudocode.

Your design must ensure four things:

1. every source maps to its own record group, even if that group is empty;
2. final records form contiguous ranges in source order;
3. each source index records where its range starts and how long it is;
4. runtime lookup uses only (start, count) and does not rescan the original edge list.

If build_source_index() feels too large to write immediately, first draw “source 0 group, source 1 group, source 2 group” on paper and then think about how those groups concatenate into records.

## Part A: build source index and records

### What does this function do?

`build_source_index()` converts a raw edge list into two data structures suitable for later source-based sequential access:

1. `source_index`: tells us where one source's records begin and how many there are;
2. `records`: stores the actual target and weight entries contiguously.

### Inputs

- `num_sources`: total number of sources; valid source IDs are `0 ... num_sources-1`;
- `edges`: raw connectivity list, where each edge is  
  `(source, target, weight)`.

### Outputs

Return **two lists** in this fixed order:

`(source_index, records)`

where:

1. `source_index` has length `num_sources`.  
   `source_index[source] = (start, count)`:
   - `start` is the position of this source's first record in `records`;
   - `count` is the number of outgoing synapses for this source.
2. `records` is a contiguous list of `(target, weight)` entries.  
   Records belonging to the same source must be adjacent and preserve that source's original input-edge order.

Even a source with zero outgoing synapses still needs a `(start, 0)` entry in `source_index`.

In [ ]:
Edge = tuple[int, int, int]   # (source, target, weight)
Record = tuple[int, int]      # (target, weight)


def build_source_index(
    num_sources: int,
    edges: list[Edge],
) -> tuple[list[tuple[int, int]], list[Record]]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: pack edges by source")
    # YOUR CODE ENDS HERE
    return source_index, records

## Part B: look up one source by (start, count)

### What does this function do?

`lookup_source()` receives one `source_id` at event time and uses the already-built `source_index` to locate that source's contiguous range in `records`.

It must not rescan the original edge list.

### Inputs

- `source_id`: source that produced the current spike;
- `source_index`: `(start, count)` index created in Part A;
- `records`: contiguous `(target, weight)` records created in Part A.

### Output

Return **one list of records**:

- containing only the `(target, weight)` entries for this `source_id`;
- preserving their order in `records`;
- returning an empty list `[]` when this source has `count = 0`.

The output is one source's slice, not the complete records store.

In [ ]:
def lookup_source(
    source_id: int,
    source_index: list[tuple[int, int]],
    records: list[Record],
) -> list[Record]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: slice by start and count")
    # YOUR CODE ENDS HERE

## Check your implementation

The grader checks exact ranges, zero fanout, lookup slices, and record order within each source.

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson11 import check

check(
    build_source_index=build_source_index,
    lookup_source=lookup_source,
    language="en",
)

## Human Check

Use your own implementation for fault diagnosis:

1. build_source_index() is build-time preprocessing, while lookup_source() is event-time lookup. Why must the latter not rescan the original edge list?
2. If one source's lookup accidentally includes the first record of the next source, would you inspect start, count, or the target accumulator first, and why?
3. When a zero-fanout source has count=0, what meaning does start still carry?
4. Point to your lookup_source(): why does it not need the original source_id field for each record?
5. Does this representation guarantee that every memory access is always faster, or does it specifically reduce how many records must be examined per spike? Explain the distinction.